- [x] Refresh build_user_vector to encode raw user inputs with saved label encoders
- [x] Verify defaults/fallbacks pull from listings when values are missing
- [x] Update knn_query usage example to show encoded constraint handling

# ETL listings

* Extraccion de los datos de la url
* Transformación de los datos, los datos deben estar numerico, los modelos codificados se guardan para su consulta posterior

In [770]:
import numpy as np
import pandas as pd
import warnings

import seaborn as sns
import matplotlib.pyplot as plt

In [771]:
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_style("ticks")
odx = pd.IndexSlice
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

%config InlineBackend.figure_format = 'retina'
%matplotlib inline

## Descarga de los datos

In [772]:
def read_url(link):
    """ Creates a pandas DataFrame from data online
    - Parameters:
        - link: link to the zipped data
    - Returns:
    """
    import io
    import requests
    import pandas as pd

    # Define URL and extract information
    response = requests.get(link)
    content = response.content
    # Convert into a Pandas DataFrame
    df = pd.read_csv(io.BytesIO(content), sep=',', compression='gzip')

    return df

listings = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2025-06-25/data/listings.csv.gz')
print(listings.shape)

(26401, 79)


In [773]:
listings[listings['id'].isna()]

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


## Analisis exploratorio de los datos

In [774]:
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,35797,https://www.airbnb.com/rooms/35797,20250625031918,2025-06-26,city scrape,Villa Dante,"Dentro de Villa un estudio de arte con futon, ...","Santa Fe Shopping Mall, Interlomas Park and th...",https://a0.muscache.com/pictures/f395ab78-1185...,153786,https://www.airbnb.com/users/show/153786,Dici,2010-06-28,"Mexico City, Mexico","Master in visual arts, film photography & Mark...",NaN,NaN,NaN,f,https://a0.muscache.com/im/pictures/user/00de1...,https://a0.muscache.com/im/pictures/user/00de1...,NaN,1.00,1.00,"['email', 'phone', 'work_email']",t,t,"Mexico City, D.f., Mexico",Cuajimalpa de Morelos,NaN,19.38,-99.27,Entire villa,Entire home/apt,2,1.00,1 bath,1.00,1.00,"[""Kitchen"", ""Resort access"", ""Hot water"", ""Cou...","$3,799.00",1,7,1.00,1.00,7.00,7.00,1.00,7.00,NaN,t,29,59,89,364,2025-06-26,0,0,0,188,0,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,f,1,1,0,0,NaN
1,44616,https://www.airbnb.com/rooms/44616,20250625031918,2025-07-01,city scrape,Condesa Haus,A new concept of hosting in mexico through a b...,NaN,https://a0.muscache.com/pictures/251410/ec75fe...,196253,https://www.airbnb.com/users/show/196253,Fernando,2010-08-09,"Mexico City, Mexico",Condesa Haus Rentals offers independent stud...,within an hour,100%,91%,f,https://a0.muscache.com/im/users/196253/profil...,https://a0.muscache.com/im/users/196253/profil...,Condesa,13.00,13.00,"['email', 'phone', 'work_email']",t,t,NaN,Cuauhtémoc,NaN,19.41,-99.18,Entire home,Entire home/apt,14,5.50,5.5 baths,5.00,8.00,"[""Free street parking"", ""Free parking on premi...","$18,000.00",1,180,1.00,1.00,180.00,180.00,1.00,180.00,NaN,t,29,59,89,360,2025-07-01,65,1,0,179,0,6,108000.00,2011-11-09,2025-01-01,4.59,4.56,4.70,4.87,4.78,4.98,4.47,NaN,f,9,4,2,0,0.39
2,56074,https://www.airbnb.com/rooms/56074,20250625031918,2025-07-01,city scrape,Great space in historical San Rafael,This great apartment is located in one of the ...,Very traditional neighborhood with all service...,https://a0.muscache.com/pictures/3005118/60dac...,265650,https://www.airbnb.com/users/show/265650,Maris,2010-10-19,"Mexico City, Mexico",I am a University Professor now retired after ...,within a few hours,100%,100%,f,https://a0.muscache.com/im/users/265650/profil...,https://a0.muscache.com/im/users/265650/profil...,San Rafael,1.00,5.00,"['email', 'phone']",t,t,"Mexico City, DF, Mexico",Cuauhtémoc,NaN,19.44,-99.16,Entire condo,Entire home/apt,2,1.00,1 bath,1.00,1.00,"[""Dining table"", ""Hot water"", ""Hangers"", ""Esse...",$585.00,15,250,15.00,15.00,250.00,250.00,15.00,250.00,NaN,t,3,33,63,338,2025-07-01,84,1,0,157,0,30,17550.00,2

### Limpieza de los datos

In [775]:
# Convert price to numeric (remove $ and ,)
listings['price'] = listings['price'].replace('[\$,]', '', regex=True).astype(float)

df_nulls = pd.DataFrame(listings.isnull().sum() / listings.shape[0] * 100).sort_values(by=0, ascending=False)
df_nulls[df_nulls[0] > 0].T

,calendar_updated,neighbourhood_group_cleansed,license,host_neighbourhood,neighbourhood,neighborhood_overview,host_about,host_location,host_response_rate,host_response_time,host_acceptance_rate,review_scores_communication,review_scores_value,review_scores_checkin,review_scores_cleanliness,review_scores_accuracy,review_scores_location,review_scores_rating,last_review,first_review,reviews_per_month,beds,bathrooms,estimated_revenue_l365d,price,host_is_superhost,has_availability,bedrooms,host_since,host_listings_count,host_picture_url,host_verifications,host_total_listings_count,host_thumbnail_url,host_identity_verified,host_has_profile_pic,host_name,description,bathrooms_text,maximum_maximum_nights,minimum_maximum_nights,maximum_minimum_nights,minimum_minimum_nights
0,100.00,100.00,100.00,48.60,47.09,47.09,42.56,23.62,17.75,17.75,13.67,12.78,12.78,12.78,12.78,12.78,12.78,12.78,12.78,12.78,12.78,12.49,12.43,12.40,12.40,5.15,3.79,3.50,3.46,3.46,3.46,3.46,3.46,3.46,3.46,3.46,3.25,2.91,0.12,0.08,0.08,0.08,0.08


## variables informativas

In [776]:
info_cols = [
    'id',
    'name',
    'description',
    'neighborhood_overview',
    'listing_url'
]

In [777]:
listings[info_cols]

,id,name,description,neighborhood_overview,listing_url
0,35797,Villa Dante,"Dentro de Villa un estudio de arte con futon, ...","Santa Fe Shopping Mall, Interlomas Park and th...",https://www.airbnb.com/rooms/35797
1,44616,Condesa Haus,A new concept of hosting in mexico through a b...,NaN,https://www.airbnb.com/rooms/44616
2,56074,Great space in historical San Rafael,This great apartment is located in one of the ...,Very traditional neighborhood with all service...,https://www.airbnb.com/rooms/56074
3,67703,"2 bedroom apt. deco bldg, Condesa","Comfortably furnished, sunny, 2 bedroom apt., ...",NaN,https://www.airbnb.com/rooms/67703
4,70644,Beautiful light Studio Coyoacan- full equipped !,COYOACAN designer studio quiet & safe! well eq...,Coyoacan is a beautiful neighborhood famous fo...,https://www.airbnb.com/rooms/70644
...,...,...,...,...,...
26396,1450299137475327273,Central 68 Arena CDMX Aduana Pantaco Ind Vallejo,New 60-meter luxury apartment with two spaciou...,NaN,https://www.airbnb.com/rooms/1450299137475327273
26397,1450300528106131951,CDMX | Business Class Flat,Experience elevated business travel in our bea...,NaN,https://www.airbnb.com/rooms/1450300528106131951
26398,1450345027980478127,Corazón CDMX Roma norte/Reforma,"Apartment in the heart of Mexico City, 2 minut...",NaN,https://www.airbnb.com/rooms/1450345027980478127
26399,1450394952972090126,Chic Loft + Lap Pool & Gym,Stay in a chic loft inside a restored historic...,NaN,https://www.airbnb.com/rooms/1450394952972090126


### Precio del host

In [778]:
listings["price"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

count    23127.00
mean      1989.29
std      18500.68
min         66.00
1%         218.00
5%         329.00
25%        630.00
50%       1039.00
75%       1647.00
95%       3970.00
99%      10000.00
max     900000.00
Name: price, dtype: float64

In [779]:
# Rows with price null
listings['price'].isna().sum()

3274

In [780]:
# Drop rows with null price
listings = listings[~listings['price'].isna()]

In [781]:
listings.shape

(23127, 79)

### Factores del host

In [782]:
# listings.filter(like='host_').columns
host_factors = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    #'host_since_delta',
    'host_since', # new feature: days since host joined
    'host_response_rate',
    'host_acceptance_rate',
    'host_total_listings_count',
    'host_listings_count'
]

In [783]:
listings[host_factors].isnull().sum() / listings[host_factors].shape[0] * 100

host_is_superhost            5.27
host_has_profile_pic         3.84
host_identity_verified       3.84
host_since                   3.84
host_response_rate          12.51
host_acceptance_rate         9.44
host_total_listings_count    3.84
host_listings_count          3.84
dtype: float64

In [784]:
listings["host_is_superhost"] = listings["host_is_superhost"].map({'t': 1, 'f': 0})
listings["host_is_superhost"] = listings["host_is_superhost"].fillna(2) # 2 means unknown

listings["host_has_profile_pic"] = listings["host_has_profile_pic"].map({'t': 1, 'f': 0})
listings["host_has_profile_pic"] = listings["host_has_profile_pic"].fillna(2) # 2 means unknown

listings["host_identity_verified"] = listings["host_identity_verified"].map({'t': 1, 'f': 0})
listings["host_identity_verified"] = listings["host_identity_verified"].fillna(2) # 2 means unknown

listings["host_since"]  = pd.to_datetime(listings["host_since"], errors='coerce') # create delta days from today
listings["host_since_delta"] = (pd.Timestamp('2025-06-25') - listings["host_since"]).dt.days
listings["host_since_delta"] = listings["host_since_delta"].fillna(listings["host_since_delta"].median())

listings["host_response_rate"] = listings["host_response_rate"].str.replace('%', '').astype(float)
listings["host_response_rate"] = listings["host_response_rate"].fillna(listings["host_response_rate"].median())

listings["host_acceptance_rate"] = listings["host_acceptance_rate"].str.replace('%', '').astype(float)
listings["host_acceptance_rate"] = listings["host_acceptance_rate"].fillna(listings["host_acceptance_rate"].median())

listings["host_total_listings_count"] = listings["host_total_listings_count"].fillna(1) # assume 1 because they have a listing

listings["host_listings_count"] = listings["host_listings_count"].fillna(1) # assume 1 because they have a listing

In [785]:
host_factors = [  x if x != "host_since" else "host_since_delta" for x in host_factors ]
host_factors

['host_is_superhost',
 'host_has_profile_pic',
 'host_identity_verified',
 'host_since_delta',
 'host_response_rate',
 'host_acceptance_rate',
 'host_total_listings_count',
 'host_listings_count']

### Factores de funcion

In [786]:
function_factors = [
    'price',
    'accommodates', # number of guests
    'bathrooms',
    'bedrooms',
    'beds',
    'property_type',
    'amenities',
    'room_type',
    'calculated_host_listings_count',
    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms'
]

In [787]:
listings[function_factors].isna().sum() / listings[function_factors].shape[0] * 100

price                                          0.00
accommodates                                   0.00
bathrooms                                      0.03
bedrooms                                       0.35
beds                                           0.10
property_type                                  0.00
amenities                                      0.00
room_type                                      0.00
calculated_host_listings_count                 0.00
calculated_host_listings_count_entire_homes    0.00
calculated_host_listings_count_private_rooms   0.00
calculated_host_listings_count_shared_rooms    0.00
dtype: float64

In [788]:
def extract_amenities(amenities_str):
    amenities_str = str(amenities_str)
    amenities_list = []
    if pd.isna(amenities_str):
        return None
    amenities_str = amenities_str.replace('"', '').replace('[', '').replace(']', '')
    amenities_list = [amenity.strip() for amenity in amenities_str.split(',')]
    return len(amenities_list)

In [789]:
listings["bathrooms"] = listings["bathrooms"].fillna(0) # shared
listings["bedrooms"] = listings["bedrooms"].fillna(0) # shared bedroom
listings["beds"] = listings["beds"].fillna(1) # at least one bed

listings["amenities_n"] = listings["amenities"].map(lambda x: extract_amenities(x))
listings["bathrooms"] = listings["bathrooms"].astype(int)
listings["bedrooms"] = listings["bedrooms"].astype(int)
listings["beds"] = listings["beds"].astype(int)

listings['amenities_parsed'] = listings['amenities'].apply(lambda x: str(x).strip('{}').replace('"', '').replace('[', '').replace(']', '').split(', '))
# convert list to str before save the database
listings['amenities_parsed'] = listings['amenities_parsed'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')

# Cambio a variables nuevas
function_factors = [x if x != "amenities" else "amenities_n" for x in function_factors]
function_factors.append('amenities_parsed')

### Factores de reputacion

In [790]:
reputation_factors = [
    'number_of_reviews',
    'reviews_per_month',
    'review_scores_rating', # this is general
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    'number_of_reviews_ltm',
    'number_of_reviews_l30d',
    'first_review', # delta
    'last_review' # delta
]

In [791]:
listings['reviews_per_month'] = listings['reviews_per_month'].fillna(listings['reviews_per_month'].median())
listings['review_scores_rating'] = listings['review_scores_rating'].fillna(listings['review_scores_rating'].median())
listings['review_scores_accuracy'] = listings['review_scores_accuracy'].fillna(listings['review_scores_accuracy'].median())
listings['review_scores_cleanliness'] = listings['review_scores_cleanliness'].fillna(listings['review_scores_cleanliness'].median())
listings['review_scores_checkin'] = listings['review_scores_checkin'].fillna(listings['review_scores_checkin'].median())
listings['review_scores_communication'] = listings['review_scores_communication'].fillna(listings['review_scores_communication'].median())
listings['review_scores_location'] = listings['review_scores_location'].fillna(listings['review_scores_location'].median())
listings['review_scores_value'] = listings['review_scores_value'].fillna(listings['review_scores_value'].median())
listings['reviews_per_month'] = listings['reviews_per_month'].fillna(listings['reviews_per_month'].median())

listings["first_review"]  = pd.to_datetime(listings["first_review"], errors='coerce') # create delta days from today
listings["first_review"] = (pd.Timestamp('2025-06-25') - listings["first_review"]).dt.days
listings["first_review"] = listings["first_review"].fillna(listings["first_review"].median())

listings["last_review"]  = pd.to_datetime(listings["last_review"], errors='coerce') # create delta days from today
listings["last_review"] = (pd.Timestamp('2025-06-25') - listings["last_review"]).dt.days
listings["last_review"] = listings["last_review"].fillna(listings["last_review"].median())

In [792]:
listings[reputation_factors].isnull().sum() / listings[reputation_factors].shape[0] * 100

number_of_reviews             0.00
reviews_per_month             0.00
review_scores_rating          0.00
review_scores_accuracy        0.00
review_scores_cleanliness     0.00
review_scores_checkin         0.00
review_scores_communication   0.00
review_scores_location        0.00
review_scores_value           0.00
number_of_reviews_ltm         0.00
number_of_reviews_l30d        0.00
first_review                  0.00
last_review                   0.00
dtype: float64

In [793]:
listings["number_of_reviews"] = listings["number_of_reviews"].astype(int)
listings["reviews_per_month"] = listings["reviews_per_month"].astype(int)
listings["number_of_reviews_ltm"] = listings["number_of_reviews_ltm"].astype(int)
listings["number_of_reviews_l30d"] = listings["number_of_reviews_l30d"].astype(int)
listings["first_review"] = listings["first_review"].astype(int)
listings["last_review"] = listings["last_review"].astype(int)
listings["reviews_per_month"] = listings["reviews_per_month"].astype(int)

listings["review_scores_rating"] = listings["review_scores_rating"].astype(float)
listings["review_scores_accuracy"] = listings["review_scores_accuracy"].astype(float)
listings["review_scores_cleanliness"] = listings["review_scores_cleanliness"].astype(float)
listings["review_scores_checkin"] = listings["review_scores_checkin"].astype(float)
listings["review_scores_communication"] = listings["review_scores_communication"].astype(float)
listings["review_scores_location"] = listings["review_scores_location"].astype(float)
listings["review_scores_value"] = listings["review_scores_value"].astype(float)
listings["review_scores_cleanliness"] = listings["review_scores_cleanliness"].astype(float)

### Fatores de ubicacion

In [794]:
location_factors = [
    'neighbourhood_cleansed',
    'latitude',
    'longitude'
]

In [795]:
listings[location_factors].isnull().sum()

neighbourhood_cleansed    0
latitude                  0
longitude                 0
dtype: int64

### Factores miscelaneos

In [796]:
misellaneous_factors = [
    'minimum_nights',
    'maximum_nights',
    'minimum_minimum_nights',
    'maximum_minimum_nights',
    'minimum_maximum_nights',
    'maximum_maximum_nights',
    'minimum_nights_avg_ntm',
    'maximum_nights_avg_ntm',
    'has_availability',
    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'instant_bookable'
]

In [797]:
listings["has_availability"] = listings["has_availability"].fillna('f')
listings["has_availability"] = listings["has_availability"].map({'t': 1, 'f': 0})
listings["instant_bookable"] = listings["instant_bookable"].map({'t': 1, 'f': 0})
listings["minimum_minimum_nights"] = listings["minimum_minimum_nights"].fillna(listings["minimum_minimum_nights"].median())
listings["maximum_minimum_nights"] = listings["maximum_minimum_nights"].fillna(listings["maximum_minimum_nights"].median())
listings["minimum_maximum_nights"] = listings["minimum_maximum_nights"].fillna(listings["minimum_maximum_nights"].median())
listings["maximum_maximum_nights"] = listings["maximum_maximum_nights"].fillna(listings["maximum_maximum_nights"].median())

In [798]:
listings[misellaneous_factors].isnull().sum()

minimum_nights            0
maximum_nights            0
minimum_minimum_nights    0
maximum_minimum_nights    0
minimum_maximum_nights    0
maximum_maximum_nights    0
minimum_nights_avg_ntm    0
maximum_nights_avg_ntm    0
has_availability          0
availability_30           0
availability_60           0
availability_90           0
availability_365          0
instant_bookable          0
dtype: int64

In [799]:
listings[ info_cols + host_factors + function_factors + reputation_factors + location_factors + misellaneous_factors ].tail()

,id,name,description,neighborhood_overview,listing_url,host_is_superhost,host_has_profile_pic,host_identity_verified,host_since_delta,host_response_rate,host_acceptance_rate,host_total_listings_count,host_listings_count,price,accommodates,bathrooms,bedrooms,beds,property_type,amenities_n,room_type,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,amenities_parsed,number_of_reviews,reviews_per_month,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,neighbourhood_cleansed,latitude,longitude,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,instant_bookable
26396,1450299137475327273,Central 68 Arena CDMX Aduana Pantaco Ind Vallejo,New 60-meter luxury apartment with two spaciou...,NaN,https://www.airbnb.com/rooms/1450299137475327273,0.00,1.00,1.00,2705.00,100.00,100.00,3.00,3.00,1080.00,6,1,2,3,Entire condo,12,Entire home/apt,1,1,0,0,"Self check-in, Carbon monoxide alarm, Washer, ...",1,1,5.00,5.00,5.00,4.00,5.00,4.00,5.00,1,1,-4,-4,Azcapotzalco,19.49,-99.17,1,365,1.00,3.00,365.00,365.00,1.20,365.00,1,29,57,84,359,1
26397,1450300528106131951,CDMX | Business Class Flat,Experience elevated business travel in our bea...,NaN,https://www.airbnb.com/rooms/1450300528106131951,2.00,1.00,1.00,1797.00,100.00,98.00,27.00,23.00,982.00,2,1,1,1,Entire rental unit,11,Entire home/apt,14,14,0,0,"Air conditioning, Kitchen, Smoke alarm, Exteri...",0,1,4.84,4.88,4.85,4.92,4.92,4.92,4.80,0,0,849,28,Benito Juárez,19.39,-99.18,1,365,1.00,1.00,365.00,365.00,1.00,365.00,1,29,59,89,364,1
26398,1450345027980478127,Corazón CDMX Roma norte/Reforma,"Apartment in the heart of Mexico City, 2 minut...",NaN,https://www.airbnb.com/rooms/1450345027980478127,1.00,1.00,1.00,325.00,100.00,97.00,6.00,6.00,1612.00,4,1,2,2,Entire rental unit,7,Entire home/apt,6,3,3,0,"Carbon monoxide alarm, First aid kit, Kitchen,...",0,1,4.84,4.88,4.85,4.92,4.92,4.92,4.80,0,0,849,28,Cuauhtémoc,19.42,-99.18,1,365,1.00,1.00,365.00,365.00,1.00,365.00,1,29,59,89,364,0
26399,1450394952972090126,Chic Loft + Lap Pool & Gym,Stay in a chic loft inside a restored historic...,NaN,https://www.airbnb.com/rooms/1450394952972090126,0.00,1.00,1.00,3865.00,100.00,100.00,1.00,1.00,627.00,2,1,1,2,Entire loft,54,Entire home/apt,1,1,0,0,"Dining table, Free street parking, Pool table,...",0,1,4.84,4.88,4.85,4.92,4.92,4.92,4.80,0,0,849,28,Cuauhtémoc,19.44,-99.16,2,1125,2.00,2.00,1125.00,1125.00,2.00,1125.00,1,6,36,66,341,0
26400,1450438287340754798,Stylish 3 BR & Terrace near Reforma,Enjoy a stylish stay in this spacious 3BR apar...,NaN,https://www.airbnb.com/rooms/1450438287340754798,0.00,1.00,1.00,1.00,100.00,99.00,1.00,1.00,3070.00,6,2,3,3,Entire rental unit,43,Entire home/apt,1,1,0,0,"Dining table, Free parking on premises, Condit...",0,1,4.84,4.88,4.85,4.92,4.92,4.92,4.80,0,0,849,28,Miguel Hidalgo,19.43,-99.18,3,120,3.00,3.00,120.00,120.00,3.00,120.00,1,30,60,90,270,0


In [800]:
listings.isna().sum()

id                                                  0
listing_url                                         0
scrape_id                                           0
last_scraped                                        0
source                                              0
name                                                0
description                                       589
neighborhood_overview                           10834
picture_url                                         0
host_id                                             0
host_url                                            0
host_name                                         832
host_since                                        887
host_location                                    5388
host_about                                       9609
host_response_time                               2893
host_response_rate                                  0
host_acceptance_rate                                0
host_is_superhost           

# Item Encoder con Structural Features

In [801]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
import joblib

In [802]:
# Choose structural columns to keep
structural_columns = host_factors + function_factors + reputation_factors + location_factors + misellaneous_factors

Binarizacion de los datos categoricos

In [803]:
listings[structural_columns].select_dtypes(exclude=['int64', 'float64']).head()

,property_type,room_type,amenities_parsed,neighbourhood_cleansed
0,Entire villa,Entire home/apt,"Kitchen, Resort access, Hot water, Courtyard v...",Cuajimalpa de Morelos
1,Entire home,Entire home/apt,"Free street parking, Free parking on premises,...",Cuauhtémoc
2,Entire condo,Entire home/apt,"Dining table, Hot water, Hangers, Essentials, ...",Cuauhtémoc
3,Entire rental unit,Entire home/apt,"Hot water, TV with standard cable, Hangers, Es...",Cuauhtémoc
4,Entire rental unit,Entire home/apt,"Varies conditioner, Dining table, Free street ...",Coyoacán


In [804]:
# all columns except amenities_parsed
categorical_columns = listings[structural_columns].select_dtypes(exclude=['int64', 'float64']).columns.tolist()
categorical_columns.remove('amenities_parsed')

In [805]:
listings[categorical_columns].select_dtypes(exclude=['int64', 'float64']).head()

,property_type,room_type,neighbourhood_cleansed
0,Entire villa,Entire home/apt,Cuajimalpa de Morelos
1,Entire home,Entire home/apt,Cuauhtémoc
2,Entire condo,Entire home/apt,Cuauhtémoc
3,Entire rental unit,Entire home/apt,Cuauhtémoc
4,Entire rental unit,Entire home/apt,Coyoacán


In [ ]:
# label categorical data and save encoders
from sklearn.preprocessing import LabelEncoder
import joblib
import pandas as pd

print(f'Categorical columns to encode: {categorical_columns}')
print("Before:", listings.shape)

for col in categorical_columns:
    lb = LabelEncoder()

    # Convert to string
    col_data = listings[col].astype(str)

    # Fit + transform
    lb.fit(col_data)
    listings[col] = lb.transform(col_data)

    # SAFELY assign new column (no merge!)
    # listings[f"{col}_encoded"] = encoder_data

    # Save encoder
    joblib.dump(lb, f'models/label_encoder_{col}.pkl')

    print(f'Added encoded column for: {col}')
    print("Current shape:", listings.shape)
    print("-----------------------------------")

print("Final:", listings.shape)

Categorical columns to encode: ['property_type', 'room_type', 'neighbourhood_cleansed']
Before: (23127, 82)
Added encoded column for: property_type
Current shape: (23127, 82)
-----------------------------------
Added encoded column for: room_type
Current shape: (23127, 82)
-----------------------------------
Added encoded column for: neighbourhood_cleansed
Current shape: (23127, 82)
-----------------------------------
Final: (23127, 82)


In [809]:
categorical_columns

['property_type', 'room_type', 'neighbourhood_cleansed']

In [812]:
# get colums encoded
columns_encoded = listings[categorical_columns].select_dtypes(exclude=['int64', 'float64']).columns

# columns encoded would be used later for modeling
columns_encoded

Index([], dtype='object')

In [815]:
# Drop amenities (text) column from structural columns
structural_columns.remove('amenities_parsed')

In [ ]:
# Fit scaler and save it
scaler = StandardScaler()
struct_matrix = scaler.fit_transform(listings[structural_columns].select_dtypes(include=['int64', 'float64']))
joblib.dump(scaler, '/models/structural_scaler.pkl')

['../models/structural_scaler.pkl']

## Guardar los datos

In [817]:
listings[structural_columns].columns

Index(['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
       'host_since_delta', 'host_response_rate', 'host_acceptance_rate',
       'host_total_listings_count', 'host_listings_count', 'price',
       'accommodates', 'bathrooms', 'bedrooms', 'beds', 'property_type',
       'amenities_n', 'room_type', 'calculated_host_listings_count',
       'calculated_host_listings_count_entire_homes',
       'calculated_host_listings_count_private_rooms',
       'calculated_host_listings_count_shared_rooms', 'number_of_reviews',
       'reviews_per_month', 'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'number_of_reviews_ltm',
       'number_of_reviews_l30d', 'first_review', 'last_review',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'minimum_nights',
       'maximum_nights', 'minimum_minimum_nights', 'maxim

In [818]:
# add info columns
structural_columns += info_cols
# add amenities_parsed
structural_columns.append('amenities_parsed')

In [819]:
listings[structural_columns].head()

,host_is_superhost,host_has_profile_pic,host_identity_verified,host_since_delta,host_response_rate,host_acceptance_rate,host_total_listings_count,host_listings_count,price,accommodates,bathrooms,bedrooms,beds,property_type,amenities_n,room_type,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,number_of_reviews,reviews_per_month,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,neighbourhood_cleansed,latitude,longitude,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,instant_bookable,id,name,description,neighborhood_overview,listing_url,amenities_parsed
0,0.00,1.00,1.00,5476.00,100.00,99.00,1.00,1.00,3799.00,2,1,1,1,21,12,0,1,1,0,0,0,1,4.84,4.88,4.85,4.92,4.92,4.92,4.80,0,0,849,28,3,19.38,-99.27,1,7,1.00,1.00,7.00,7.00,1.00,7.00,1,29,59,89,364,0,35797,Villa Dante,"Dentro de Villa un estudio de arte con futon, ...","Santa Fe Shopping Mall, Interlomas Park and th...",https://www.airbnb.com/rooms/35797,"Kitchen, Resort access, Hot water, Courtyard v..."
1,0.00,1.00,1.00,5434.00,100.00,91.00,13.00,13.00,18000.00,14,5,5,8,12,26,0,9,4,2,0,65,0,4.59,4.56,4.70,4.87,4.78,4.98,4.47,1,0,4977,175,4,19.41,-99.18,1,180,1.00,1.00,180.00,180.00,1.00,180.00,1,29,59,89,360,0,44616,Condesa Haus,A new concept of hosting in mexico through a b...,NaN,https://www.airbnb.com/rooms/44616,"Free street parking, Free parking on premises,..."
2,0.00,1.00,1.00,5363.00,100.00,100.00,5.00,1.00,585.00,2,1,1,1,8,28,0,1,1,0,0,84,0,4.87,4.95,4.88,4.98,4.94,4.76,4.79,1,0,5198,118,4,19.44,-99.16,15,250,15.00,15.00,250.00,250.00,15.00,250.00,1,3,33,63,338,0,56074,Great space in historical San Rafael,This great apartment is located in one of the ...,Very traditional neighborhood with all service...,https://www.airbnb.com/rooms/56074,"Dining table, Hot water, Hangers, Essentials, ..."
3,0.00,1.00,1.00,5286.00,100.00,47.00,4.00,3.00,1696.00,4,1,2,2,17,21,0,2,2,0,0,50,0,4.90,4.82,4.76,4.94,4.92,4.98,4.92,1,0,4969,238,4,19.41,-99.17,2,30,2.00,2.00,30.00,30.00,2.00,30.00,1,3,4,32,267,0,67703,"2 bedroom apt. deco bldg, Condesa","Comfortably furnished, sunny, 2 bedroom apt., ...",NaN,https://www.airbnb.com/rooms/67703,"Hot water, TV with standard cable, Hangers, Es..."
4,1.00,1.00,1.00,5419.00,100.00,85.00,4.00,3.00,1004.00,2,1,1,1,17,51,0,3,2,1,0,132,0,4.92,4.91,4.96,4.96,4.98,4.96,4.92,8,0,4880,179,2,19.35,-99.16,3,180,3.00,4.00,180.00,180.00,3.40,180.00,1,10,25,25,211,0,70644,Beautiful light Studio Coyoacan- full equipped !,COYOACAN designer studio quiet & safe! well eq...,Coyoacan is a beautiful neighborhood famous fo...,https://www.airbnb.com/rooms/70644,"Varies conditioner, Dining table, Free street ..."


In [ ]:
# save cleaned listings to SQLite database
import sqlite3

# Create a connection to SQLite database
conn = sqlite3.connect('db/airbnb.db')

# Save the listings DataFrame to SQLite
listings[structural_columns].to_sql('listings', conn, if_exists='replace', index=False)

# Close the connection
conn.close()
print("Listings saved to SQLite database successfully!")

Listings saved to SQLite database successfully!


In [ ]:
# test the saved data
conn = sqlite3.connect('db/airbnb.db')
df_test = pd.read_sql_query("SELECT * FROM listings LIMIT 5;", conn)
conn.close()
df_test.head()

# test with assertions
assert df_test.shape[0] == 5
assert df_test.shape[1] == len(structural_columns)

print("Data integrity check passed!")


Data integrity check passed!


# Encoder for text

## Variables de tipo texto (name, description, amenities_parsed)

Text to encode

In [822]:
import pandas as pd 
import numpy as np
import sqlite3
import joblib
import os

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler

In [ ]:
# Read from SQLite database
conn = sqlite3.connect('db/airbnb.db')
listings = pd.read_sql_query('SELECT * FROM listings', conn)
conn.close()

In [825]:
listings.shape

(23127, 56)

In [826]:
text_columns = ['name', 'description', 'amenities_parsed']
listings[text_columns]

,name,description,amenities_parsed
0,Villa Dante,"Dentro de Villa un estudio de arte con futon, ...","Kitchen, Resort access, Hot water, Courtyard v..."
1,Condesa Haus,A new concept of hosting in mexico through a b...,"Free street parking, Free parking on premises,..."
2,Great space in historical San Rafael,This great apartment is located in one of the ...,"Dining table, Hot water, Hangers, Essentials, ..."
3,"2 bedroom apt. deco bldg, Condesa","Comfortably furnished, sunny, 2 bedroom apt., ...","Hot water, TV with standard cable, Hangers, Es..."
4,Beautiful light Studio Coyoacan- full equipped !,COYOACAN designer studio quiet & safe! well eq...,"Varies conditioner, Dining table, Free street ..."
...,...,...,...
23122,Central 68 Arena CDMX Aduana Pantaco Ind Vallejo,New 60-meter luxury apartment with two spaciou...,"Self check-in, Carbon monoxide alarm, Washer, ..."
23123,CDMX | Business Class Flat,Experience elevated business travel in our bea...,"Air conditioning, Kitchen, Smoke alarm, Exteri..."
23124,Corazón CDMX Roma norte/Reforma,"Apartment in the heart of Mexico City, 2 minut...","Carbon monoxide alarm, First aid kit, Kitchen,..."
23125,Chic Loft + Lap Pool & Gym,Stay in a chic loft inside a restored historic...,"Dining table, Free street parking, Pool table,..."


In [827]:
# Fill NaN values with empty strings for the selected text columns
for col in text_columns:
    listings[col] = listings[col].fillna('')

# Concatenate the cleaned text columns into 'combined_text'
listings['combined_text'] = listings.apply(lambda row: ' '.join([str(row[col]) for col in text_columns]), axis=1)

# Display the first few rows with the new 'combined_text' column to verify
print(listings[['name', 'description', 'amenities_parsed', 'combined_text']].head(2))

           name                                        description  \
0   Villa Dante  Dentro de Villa un estudio de arte con futon, ...   
1  Condesa Haus  A new concept of hosting in mexico through a b...   

                                    amenities_parsed  \
0  Kitchen, Resort access, Hot water, Courtyard v...   
1  Free street parking, Free parking on premises,...   

                                       combined_text  
0  Villa Dante Dentro de Villa un estudio de arte...  
1  Condesa Haus A new concept of hosting in mexic...  


In [828]:
#!pip3 install -U sentence-transformers

In [829]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
text_embeddings = model.encode(listings['combined_text'].tolist(), show_progress_bar=True)
print(f"Generated embeddings with shape: {text_embeddings.shape}")

Batches:   0%|          | 0/723 [00:00<?, ?it/s]

Generated embeddings with shape: (23127, 384)


In [830]:
text_embeddings_df = pd.DataFrame(text_embeddings, index=listings.index) # DataFrame embeddings

In [831]:
text_embeddings_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383
0,0.26,0.05,0.07,0.18,0.24,0.02,-0.09,0.09,0.21,0.20,0.02,-0.08,-0.07,0.22,-0.07,-0.16,0.06,0.14,0.27,0.16,0.18,0.06,-0.15,0.10,-0.34,0.04,-0.06,0.05,-0.28,-0.20,-0.02,0.05,-0.09,-0.06,0.07,-0.09,-0.23,-0.14,-0.10,-0.06,-0.21,0.15,0.11,-0.13,0.09,0.10,-0.09,0.01,0.07,-0.08,...,0.03,0.05,0.11,-0.06,0.07,-0.29,0.08,-0.08,-0.08,-0.28,-0.15,0.02,0.08,-0.09,-0.03,-0.13,-0.06,0.22,-0.09,0.06,0.06,-0.00,-0.10,-0.11,-0.18,-0.31,0.27,-0.01,-0.02,-0.16,0.11,-0.23,0.31,0.09,-0.15,0.20,-0.16,0.08,0.40,0.07,0.17,0.11,0.07,-0.01,0.05,0.00,0.05,-0.06,-0.26,-0.06
1,0.23,0.10,-0.29,0.13,0.19,-0.04,-0.06,-0.13,0.07,0.15,0.05,0.37,0.14,0.19,0.19,-0.00,0.07,-0.12,0.15,0.04,0.36,-0.19,-0.33,0.06,0.03,-0.22,-0.05,0.21,-0.16,-0.26,0.08,0.08,0.01,0.30,0.07,-0.12,0.22,-0.25,-0.21,-0.05,-0.06,0.20,-0.07,-0.07,-0.14,-0.04,0.12,0.23,0.08,-0.48,...,0.10,-0.05,0.20,0.42,-0.11,-0.23,0.34,0.06,-0.01,0.05,0.17,0.15,-0.41,-0.30,0.28,-0.20,-0.14,0.17,-0.47,-0.49,0.09,0.05,-0.12,-0.08,-0.22,-0.31,0.44,0.10,0.02,0.13,0.02,-0.20,0.21,-0.21,0.08,0.16,-0.13,-0.14,0.46,0.18,0.04,-0.18,-0.27,-0.12,0.05,-0.03,0.25,0.22,-0.27,0.25
2,0.41,0.27,0.05,0.01,-0.07,0.13,-0.18,0.18,0.02,-0.06,0.07,-0.10,-0.00,-0.02,0.07,-0.09,-0.15,-0.01,0.09,-0.04,-0.00,-0.14,-0.10,0.28,0.00,0.32,0.06,0.37,-0.04,-0.05,0.01,0.12,0.37,0.24,0.19,0.21,0.05,-0.07,0.16,-0.08,-0.19,0.14,0.37,0.11,0.04,0.04,0.10,-0.03,0.29,-0.24,...,-0.40,-0.17,0.11,-0.22,-0.03,0.07,-0.17,-0.23,-0.03,-0.16,-0.02,0.45,-0.40,-0.08,0.07,0.10,-0.14,-0.15,-0.08,-0.19,-0.04,0.21,-0.06,0.07,-0.39,-0.59,0.25,-0.20,0.11,0.19,0.09,-0.07,0.21,-0.06,-0.20,0.29,-0.14,0.03,0.20,0.17,-0.12,0.14,-0.11,0.01,0.11,-0.07,0.08,-0.03,-0.28,0.08
3,0.01,0.03,0.03,0.21,0.25,0.19,-0.30,-0.17,-0.03,0.28,-0.10,-0.18,-0.13,0.00,0.10,0.03,0.10,0.01,0.14,0.21,0.07,-0.08,-0.03,-0.06,-0.15,0.05,-0.16,0.14,-0.25,-0.24,0.03,0.05,-0.00,0.00,0.28,-0.18,0.07,-0.08,-0.23,-0.38,-0.22,0.05,0.05,-0.05,-0.01,-0.12,-0.00,0.17,0.18,-0.08,...,0.14,0.20,-0.03,-0.07,0.07,-0.19,0.16,-0.11,0.05,0.12,0.13,0.18,0.11,0.09,0.10,-0.03,0.06,0.02,-0.13,-0.08,0.14,-0.07,-0.04,0.01,-0.01,-0.20,0.31,-0.18,-0.24,-0.06,0.15,-0.20,0.06,0.07,-0.03,-0.07,0.06,-0.03,-0.08,-0.05,-0.20,-0.21,-0.12,0.15,-0.10,0.00,-0.19,-0.13,-0.14,-0.09
4,0.28,-0.05,0.09,0.27,-0.06,0.05,-0.16,-0.01,-0.13,0.15,0.06,-0.11,0.05,0.30,0.16,-0.13,0.26,0.00,0.25,-0.08,-0.12,-0.33,0.11,0.06,0.07,0.09,-0.13,0.05,-0.05,-0.28,-0.09,0.10,0.14,-0.13,0.10,0.27,-0.12,-0.04,0.13,0.28,-0.07,0.25,0.08,0.07,0.09,0.05,-0.13,-0.07,0.03,-0.12,...,-0.03,0.06,-0.22,-0.15,0.16,-0.30,0.20,-0.15,0.06,-0.13,-0.18,0.23,0.12,0.17,0.17,-0.10,-0.24,-0.01,-0.37,0.00,-0.02,-0.16,-0.49,0.03,-0.27,-0.18,0.02,0.11,-0.07,-0.10,0.17,-0.22,0.17,-0.02,-0.07,0.18,-0.32,-0.02,-0.03,0.12,0.06,0.02,-0.10,-0.13,0.01,0.08,-0.09,-0.08,-0.12,-0.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23122,0.42,0.15,0.10,0.11,0.11,0.11,-0.00,0.23,0.31,0.28,0.25,0.20,0.12,0.04,0.23,0.04,0.19,-0.18,0.25,0.21,-0.09,-0.33,-0.26,-0.10,0.04,0.14,-0.20,0.26,-0.14,0.14,0.07,0.21,0.24,-0.11,0.09,-0.13,-0.15,-0.03,-0.11,-0.02,-0.18,0.23,-0.07,-0.18,-0.03,0.05,-0.12,0.09,0.29,0.20,...,-0.21,0.14,0.15,-0.07,0.08,-0.02,0.08,0.22,0.16,-0.05,0.17,0.23,-0.10,-0.24,-0.05,-0.12,-0.37,-0.16,-0.14,-0.10,-0.14,-0.09,-0.01,-0.24,-0.18,-0.20,0.21,-0.02,-0.14,-0.25,0.30,-0.13,-0.13,-0.34

In [ ]:
# Save data and metadata
os.makedirs('/data', exist_ok=True)
np.save('data/text_embeddings.npy', text_embeddings)

In [ ]:
# Save id mapping in sqlite, index and id
import sqlite3
conn = sqlite3.connect('db/airbnb.db')
listings.reset_index(drop=False, inplace=True)
listings['index_df'] = listings.index
listings[['index_df','id']].to_sql('listing_ids', conn, if_exists='replace')
conn.close()

## Train the model

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.neighbors import NearestNeighbors

# load data
text_embeddings = np.load('data/text_embeddings.npy')

In [ ]:
knn = NearestNeighbors(n_neighbors=5, algorithm='auto')
knn.fit(text_embeddings)

# Save the trained model
joblib.dump(knn, 'models/knn_text_embeddings.pkl')

print("NearestNeighbors model trained and saved as 'knn_text_embeddings.pkl'")

NearestNeighbors model trained and saved as 'knn_text_embeddings.pkl'


## Find listing

In [836]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import joblib

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def find_similar_listings(query_text: str, n_neighbors: int = 5):
    """
    Finds N most similar listings based on a combination of text query and property attributes.

    Args:
        query_text (str): A natural language description of desired listing features.
        query_attributes (dict): A dictionary of desired numerical and categorical attributes
                                 (e.g., {'price': 100, 'room_type': 'Private room', 'accommodates': 2}).
        n_neighbors (int): The number of similar listings to return.

    Returns:
        np.ndarray: An array of indices of the most similar listings in the original DataFrame.
    """
    
    global model

    # Generate text embedding for query_text
    query_text_embedding = model.encode(query_text).reshape(1, -1)

    # Load the saved KNN model pkl
    knn = joblib.load('/Users/gblasd/Documents/Code/SmartBnB/models/knn_text_embeddings.pkl')

    # Use the KNN model to find similar listings
    distances, indices = knn.kneighbors(query_text_embedding, n_neighbors=n_neighbors)

    # print("Distances to similar listings:")
    # print(distances)
    # print("Indices of similar listings:")
    # print(indices)

    return indices.flatten()

In [ ]:
# function to query listing by id from database sqlite
def get_listing_by_id(listing_id: list[int]) -> pd.DataFrame:
    """
    Retrieves listing details from the SQLite database based on a list of listing IDs.

    Args:
        listing_id (list[int]): A list of listing IDs to retrieve.
    Returns:
        pd.DataFrame: A DataFrame containing the listing details.
    """
    import sqlite3
    import pandas as pd

    # Create a connection to SQLite database
    conn = sqlite3.connect('db/airbnb.db')

    # Convert list of IDs to a comma-separated string
    id_tuple = tuple(listing_id)
    if len(id_tuple) == 1:
        id_tuple = (id_tuple[0], id_tuple[0])  # Ensure it's a tuple of length 2 for single ID

    # Query to get listings by IDs from listing_ids table and then from listings table
    query = f"""
    SELECT l.*
    FROM listings l
    JOIN listing_ids li ON l.id = li.id
    WHERE li.index_df IN {id_tuple}
    """ 
    
    listings_df = pd.read_sql_query(query, conn)

    # decode data for ['property_type', 'room_type', 'neighbourhood_cleansed'] columns
    # encoder path
    encoder_property_type = joblib.load('models/label_encoder_property_type.pkl')
    listings_df['property_type'] = listings_df['property_type'].map(lambda x: encoder_property_type.inverse_transform([x])[0])

    encoder_neighbourhood_cleansed = joblib.load('models/label_encoder_neighbourhood_cleansed.pkl')
    listings_df['neighbourhood_cleansed'] = listings_df['neighbourhood_cleansed'].map(lambda x: encoder_property_type.inverse_transform([x])[0])

    encoder_room_type = joblib.load('models/label_encoder_room_type.pkl')
    listings_df['room_type'] = listings_df['room_type'].map(lambda x: encoder_property_type.inverse_transform([x])[0])

    conn.close()
    return listings_df

In [855]:
# Example usage
# Define a sample query with both text and attributes
sample_query_text = "beautiful apartment with a view and good for families"

# Find similar listings
similar_listing_indices = find_similar_listings(
    query_text=sample_query_text,
    #query_attributes=sample_query_attributes,
    n_neighbors=5
)

print("Indices of 5 similar listings:", similar_listing_indices)
print("\nDetails of similar listings:")
get_listing_by_id(similar_listing_indices)[['id', 'name', 'price', 'description', 'listing_url', 'property_type', 'room_type', 'neighbourhood_cleansed']]

Indices of 5 similar listings: [20041 18684  3625 10319   820]

Details of similar listings:


,id,name,price,description,listing_url,property_type,room_type,neighbourhood_cleansed
0,13665937,Beautiful sunny apartment in trendy Condesa,4360.00,"This fantastic apartment of 1,400 square feet ...",https://www.airbnb.com/rooms/13665937,Entire rental unit,Campsite,Earthen home
1,33128669,Increíble Penthouse,2244.00,The apartment is designed and designed for fam...,https://www.airbnb.com/rooms/33128669,Entire condo,Campsite,Earthen home
2,733272273789823087,Sensational Rooftop apartment,641.00,You’ll treasure your time at this memorable ap...,https://www.airbnb.com/rooms/733272273789823087,Private room,Castle,Earthen home
3,1247117159143145962,Habitación en espectacular casa,630.00,This elegant accommodation is ideal for couple...,https://www.airbnb.com/rooms/1247117159143145962,Private room in home,Castle,Entire condo
4,1315481830889032544,Charming Condesa Apartment,493.00,Relax with the whole family at this peaceful p...,https://www.airbnb.com/rooms/1315481830889032544,Entire rental unit,Campsite,Entire chalet


In [857]:
# Example usage
# Define a sample query with both text and attributes
sample_query_text = "Departamento con roof garden y buena vista familiar"

# Find similar listings
similar_listing_indices = find_similar_listings(
    query_text=sample_query_text,
    #query_attributes=sample_query_attributes,
    n_neighbors=5
)

print("Indices of 5 similar listings:", similar_listing_indices)
print("\nDetails of similar listings:")
get_listing_by_id(similar_listing_indices)[['id', 'name', 'price', 'description', 'listing_url', 'property_type', 'room_type', 'neighbourhood_cleansed']]

Indices of 5 similar listings: [ 2843 12924 21978  1173  4063]

Details of similar listings:


,id,name,price,description,listing_url,property_type,room_type,neighbourhood_cleansed
0,16428350,Loft Luminoso y Amplio con Excelentes Vistas e...,1353.00,"I am changing conditions, so one small family ...",https://www.airbnb.com/rooms/16428350,Entire loft,Campsite,Entire cottage
1,27877474,LOFT en Ciudad Jardín,540.00,Fun modern loft. You'll find a skylight in the...,https://www.airbnb.com/rooms/27877474,Entire loft,Campsite,Castle
2,35878331,Experiencia Única en área de Roof Garden / Ref...,820.00,None,https://www.airbnb.com/rooms/35878331,Tiny home,Campsite,Earthen home
3,906458615813509727,Departamento con roof garden - Centro histórico,1760.00,"From this central home, the whole group will h...",https://www.airbnb.com/rooms/906458615813509727,Entire serviced apartment,Campsite,Earthen home
4,1407990362694026025,Unique Designer Oasis in Condesa. Best roof ga...,11539.00,Our home is where art and architecture blend. ...,https://www.airbnb.com/rooms/1407990362694026025,Entire home,Campsite,Earthen home


In [859]:
# Example usage
# Define a sample query with both text and attributes
sample_query_text = "roof in polanco"

# Find similar listings
similar_listing_indices = find_similar_listings(
    query_text=sample_query_text,
    #query_attributes=sample_query_attributes,
    n_neighbors=5
)

print("Indices of 5 similar listings:", similar_listing_indices)
print("\nDetails of similar listings:")
get_listing_by_id(similar_listing_indices)[['id', 'name', 'price', 'description', 'listing_url', 'property_type', 'room_type', 'neighbourhood_cleansed']]

Indices of 5 similar listings: [ 5234 16161 14663 14660  6851]

Details of similar listings:


,id,name,price,description,listing_url,property_type,room_type,neighbourhood_cleansed
0,42272228,Rooftop w/Mega Views+Pool | Refined Stay w/Bal...,2847.00,Stay at this stunning ULIV apartment and unloc...,https://www.airbnb.com/rooms/42272228,Entire rental unit,Campsite,Entire cottage
1,49713023,ET_2103 Loft con Roof en Nuevo Polanco,1710.00,"Loft in a condominium, with independent access...",https://www.airbnb.com/rooms/49713023,Entire condo,Campsite,Entire cottage
2,1013782192911978502,Modern Apt in Polanco! | Chic Terrace & Rooftop,2877.00,Enjoy the beauty of Polanco at our fantastic b...,https://www.airbnb.com/rooms/1013782192911978502,Entire rental unit,Campsite,Entire cottage
3,1013812164238597295,PH w/Pvt Terrace | Prime Location+Elegant Rooftop,3491.00,This wonderful Penthouse has it all: a beautif...,https://www.airbnb.com/rooms/1013812164238597295,Entire rental unit,Campsite,Entire cottage
4,1111705711096746137,w* | Perfect Loft with Terrace in Polanco,1503.00,This completely remodeled 1-bedroom Loft is lo...,https://www.airbnb.com/rooms/1111705711096746137,Entire rental unit,Campsite,Entire cottage


# Fusionar text embeddings + structural features - Item Encoder

In [ ]:
# ml/encoders/item_encoder.py
import numpy as np
from pathlib import Path
import joblib
import sqlite3

def build_item_embeddings():
    """
    Build item embeddings by concatenating text embeddings and structural features.
    """

    # load embeddings
    text_emb = np.load("data/text_embeddings.npy")
    
    # load structural features from database sqlite
    conn = sqlite3.connect('db/airbnb.db')
    struct = pd.read_sql_query('SELECT * FROM listings', conn)
    ids = pd.read_sql_query('SELECT * FROM listing_ids', conn)
    conn.close()


    assert text_emb.shape[0] == struct.shape[0] == ids.shape[0], "text_emb, struct and ids must have the same number of rows"

    # Concatenate all except info columns in var info_cols
    info_cols = [
        'id',
        'name',
        'description',
        'neighborhood_overview',
        'listing_url',
        'amenities_parsed'
    ]
    fused = np.concatenate([text_emb, struct.drop(columns=info_cols).values], axis=1)
    print(f"Fused embeddings shape: {fused.shape}")

    # Save fused embeddings
    np.save("data/item_embeddings.npy", fused)

build_item_embeddings()

Fused embeddings shape: (23127, 434)


In [ ]:
# Save embeddings to a .npy file
item_embeddings = np.load("data/item_embeddings.npy")
item_embeddings.shape

(23127, 434)

In [862]:
pd.DataFrame(item_embeddings)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433
0,0.26,0.05,0.07,0.18,0.24,0.02,-0.09,0.09,0.21,0.20,0.02,-0.08,-0.07,0.22,-0.07,-0.16,0.06,0.14,0.27,0.16,0.18,0.06,-0.15,0.10,-0.34,0.04,-0.06,0.05,-0.28,-0.20,-0.02,0.05,-0.09,-0.06,0.07,-0.09,-0.23,-0.14,-0.10,-0.06,-0.21,0.15,0.11,-0.13,0.09,0.10,-0.09,0.01,0.07,-0.08,...,0.00,1.00,1.00,5476.00,100.00,99.00,1.00,1.00,3799.00,2.00,1.00,1.00,1.00,21.00,12.00,0.00,1.00,1.00,0.00,0.00,0.00,1.00,4.84,4.88,4.85,4.92,4.92,4.92,4.80,0.00,0.00,849.00,28.00,3.00,19.38,-99.27,1.00,7.00,1.00,1.00,7.00,7.00,1.00,7.00,1.00,29.00,59.00,89.00,364.00,0.00
1,0.23,0.10,-0.29,0.13,0.19,-0.04,-0.06,-0.13,0.07,0.15,0.05,0.37,0.14,0.19,0.19,-0.00,0.07,-0.12,0.15,0.04,0.36,-0.19,-0.33,0.06,0.03,-0.22,-0.05,0.21,-0.16,-0.26,0.08,0.08,0.01,0.30,0.07,-0.12,0.22,-0.25,-0.21,-0.05,-0.06,0.20,-0.07,-0.07,-0.14,-0.04,0.12,0.23,0.08,-0.48,...,0.00,1.00,1.00,5434.00,100.00,91.00,13.00,13.00,18000.00,14.00,5.00,5.00,8.00,12.00,26.00,0.00,9.00,4.00,2.00,0.00,65.00,0.00,4.59,4.56,4.70,4.87,4.78,4.98,4.47,1.00,0.00,4977.00,175.00,4.00,19.41,-99.18,1.00,180.00,1.00,1.00,180.00,180.00,1.00,180.00,1.00,29.00,59.00,89.00,360.00,0.00
2,0.41,0.27,0.05,0.01,-0.07,0.13,-0.18,0.18,0.02,-0.06,0.07,-0.10,-0.00,-0.02,0.07,-0.09,-0.15,-0.01,0.09,-0.04,-0.00,-0.14,-0.10,0.28,0.00,0.32,0.06,0.37,-0.04,-0.05,0.01,0.12,0.37,0.24,0.19,0.21,0.05,-0.07,0.16,-0.08,-0.19,0.14,0.37,0.11,0.04,0.04,0.10,-0.03,0.29,-0.24,...,0.00,1.00,1.00,5363.00,100.00,100.00,5.00,1.00,585.00,2.00,1.00,1.00,1.00,8.00,28.00,0.00,1.00,1.00,0.00,0.00,84.00,0.00,4.87,4.95,4.88,4.98,4.94,4.76,4.79,1.00,0.00,5198.00,118.00,4.00,19.44,-99.16,15.00,250.00,15.00,15.00,250.00,250.00,15.00,250.00,1.00,3.00,33.00,63.00,338.00,0.00
3,0.01,0.03,0.03,0.21,0.25,0.19,-0.30,-0.17,-0.03,0.28,-0.10,-0.18,-0.13,0.00,0.10,0.03,0.10,0.01,0.14,0.21,0.07,-0.08,-0.03,-0.06,-0.15,0.05,-0.16,0.14,-0.25,-0.24,0.03,0.05,-0.00,0.00,0.28,-0.18,0.07,-0.08,-0.23,-0.38,-0.22,0.05,0.05,-0.05,-0.01,-0.12,-0.00,0.17,0.18,-0.08,...,0.00,1.00,1.00,5286.00,100.00,47.00,4.00,3.00,1696.00,4.00,1.00,2.00,2.00,17.00,21.00,0.00,2.00,2.00,0.00,0.00,50.00,0.00,4.90,4.82,4.76,4.94,4.92,4.98,4.92,1.00,0.00,4969.00,238.00,4.00,19.41,-99.17,2.00,30.00,2.00,2.00,30.00,30.00,2.00,30.00,1.00,3.00,4.00,32.00,267.00,0.00
4,0.28,-0.05,0.09,0.27,-0.06,0.05,-0.16,-0.01,-0.13,0.15,0.06,-0.11,0.05,0.30,0.16,-0.13,0.26,0.00,0.25,-0.08,-0.12,-0.33,0.11,0.06,0.07,0.09,-0.13,0.05,-0.05,-0.28,-0.09,0.10,0.14,-0.13,0.10,0.27,-0.12,-0.04,0.13,0.28,-0.07,0.25,0.08,0.07,0.09,0.05,-0.13,-0.07,0.03,-0.12,...,1.00,1.00,1.00,5419.00,100.00,85.00,4.00,3.00,1004.00,2.00,1.00,1.00,1.00,17.00,51.00,0.00,3.00,2.00,1.00,0.00,132.00,0.00,4.92,4.91,4.96,4.96,4.98,4.96,4.92,8.00,0.00,4880.00,179.00,2.00,19.35,-99.16,3.00,180.00,3.00,4.00,180.00,180.00,3.40,180.00,1.00,10.00,25.00,25.00,211.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23122,0.42,0.15,0.10,0.11,0.11,0.11,-0.00,0.23,0.31,0.28,0.25,0.20,0.12,0.04,0.23,0.04,0.19,-0.18,0.25,0.21,-0.09,-0.33,-0.26,-0.10,0.04,0.14,-0.20,0.26,-0.14,0.14,0.07,0.21,0.24,-0.11,0.09,-0.13,-0.15,-0.03,-0.11,-0.02,-0.18,0.23,-0.07,-0.18,-0.03,0.05,-0.12,0.09,0.29,0.20,...,0.00,1.00,1.00,2705.00,100.00,100.00,3.00,3.00,1080.00,6.00,1.00,2.00,3.00,8.00,12.00,0.00,1.00,1.00,0.00,0.00,1.00,1.00,5.00,5.00,5.00,4.00,5.00,4.00,5.00,1.00,1.00,-4.00,-4

## Indexar con KNN

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
import joblib

def build_knn(n_neighbors=10, metric='cosine') -> str:
    item_embeddings = np.load("data/item_embeddings.npy")

    # build and train model
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric=metric, algorithm='auto', n_jobs=-1)
    nn.fit(item_embeddings)

    # save the model
    joblib.dump(nn, "models/knn_index.pkl")
    
    return "models/knn_index.pkl"

build_knn(10, 'cosine')

'/Users/gblasd/Documents/Code/SmartBnB/models/knn_index.pkl'

## Ranker tabular con XGBoost

In [891]:
# Ensure xgboost is available
try:
    import xgboost as xgb
except ImportError:
    %pip install --quiet xgboost
    import xgboost as xgb
from xgboost import XGBRanker

In [ ]:
from pathlib import Path
import joblib

# Build training data for the tabular ranker
ranker_feature_cols = [
    col for col in structural_columns
    if col not in info_cols and col != 'amenities_parsed' and col in listings.columns
]
ranker_feature_cols = list(dict.fromkeys(ranker_feature_cols))
ranker_target_col = 'review_scores_rating'
group_column = 'neighbourhood_cleansed'

if ranker_target_col not in listings.columns:
    raise KeyError(f"No se encontró la columna objetivo '{ranker_target_col}' en listings.")
if group_column not in listings.columns:
    raise KeyError(f"No se encontró la columna de agrupación '{group_column}' en listings.")

selected_columns = list(dict.fromkeys(ranker_feature_cols + [ranker_target_col, group_column]))
ranker_data = listings[selected_columns].copy()
ranker_data = ranker_data.dropna(subset=[ranker_target_col])

# Mantener únicamente grupos con al menos dos elementos para entrenamiento del ranker
group_counts = ranker_data[group_column].value_counts()
valid_groups = group_counts[group_counts > 1].index
ranker_data = ranker_data[ranker_data[group_column].isin(valid_groups)].sort_values(group_column).reset_index(drop=True)

if ranker_data.empty:
    raise ValueError("No hay suficientes grupos con más de un elemento para entrenar el ranker.")

ranker_feature_cols = [col for col in ranker_feature_cols if col != ranker_target_col]

def _ranker_default(series):
    if series.isnull().all():
        return 0.0
    if np.issubdtype(series.dtype, np.number):
        candidate = series.median(skipna=True)
        if np.isnan(candidate):
            candidate = series.mean(skipna=True)
        return float(candidate if not np.isnan(candidate) else 0.0)
    mode = series.mode(dropna=True)
    if not mode.empty:
        try:
            return float(mode.iloc[0])
        except (TypeError, ValueError):
            return 0.0
    return 0.0

defaults = {col: _ranker_default(listings[col]) for col in ranker_feature_cols}

X_ranker = ranker_data[ranker_feature_cols].fillna(defaults).to_numpy(dtype=float)
y_ranker = ranker_data[ranker_target_col].to_numpy(dtype=float)
group_sizes = ranker_data.groupby(group_column, sort=False).size().astype(int).tolist()

xgb_ranker = XGBRanker(
    objective='rank:pairwise',
    learning_rate=0.1,
    max_depth=6,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    tree_method='hist',
    eval_metric='ndcg@10',
)

xgb_ranker.fit(X_ranker, y_ranker, group=group_sizes)

ranker_bundle = {
    "model": xgb_ranker,
    "feature_columns": ranker_feature_cols,
    "target_column": ranker_target_col,
    "group_column": group_column,
    "defaults": defaults,
}

ranker_path = Path("models/tabular_ranker_xgb.joblib")
joblib.dump(ranker_bundle, ranker_path, compress="lz4")

print(f"Ranker entrenado y guardado en: {ranker_path}")
print(f"Total de registros usados: {len(ranker_data)} en {len(group_sizes)} grupos")

Ranker entrenado y guardado en: /Users/gblasd/Documents/Code/SmartBnB/models/tabular_ranker_xgb.joblib
Total de registros usados: 23127 en 16 grupos


## Query 

In [ ]:
import numpy as np
import joblib
import os
from pathlib import Path
from functools import lru_cache

try:
    import xgboost as xgb  # noqa: F401
except ImportError:
    xgb = None

# Construir vector de usuario (texto + constraints estructurales, compatible con inputs parciales)

_LABEL_ENCODER_PATHS = [
    Path("../models"),
    Path("./models"),
]
RAW_ENCODER_COLUMNS = {"property_type", "room_type", "neighbourhood_cleansed"}

STRUCT_COLUMNS_FOR_MODEL = [
    col for col in structural_columns
    if col not in info_cols and col not in {"amenities_parsed"}
    and col in listings.columns
]

_RANKER_MODEL_PATHS = [
    Path("models/tabular_ranker_xgb.joblib"),
    Path("models/tabular_ranker_xgb.joblib"),
    Path("../models/tabular_ranker_xgb.joblib"),
    Path("./models/tabular_ranker_xgb.joblib"),
]
_RANKER_CACHE: dict | None = None

@lru_cache(maxsize=None)
def _load_label_encoder(column_name: str):
    """Load LabelEncoder for a given column name from disk."""
    for directory in _LABEL_ENCODER_PATHS:
        encoder_path = directory / f"label_encoder_{column_name}.pkl"
        if encoder_path.exists():
            return joblib.load(encoder_path)
    raise FileNotFoundError(f"LabelEncoder para '{column_name}' no encontrado en {_LABEL_ENCODER_PATHS}.")


def _default_for_column(series):
    """Return a numeric default value for a structural column."""
    if pd.api.types.is_numeric_dtype(series):
        if series.empty:
            return 0.0
        mean_val = float(series.mean(skipna=True))
        if np.isnan(mean_val):
            median_val = float(series.median(skipna=True))
            mean_val = median_val if not np.isnan(median_val) else 0.0
        return mean_val
    mode_series = series.mode(dropna=True)
    if not mode_series.empty:
        try:
            return float(mode_series.iloc[0])
        except (TypeError, ValueError):
            return 0.0
    return 0.0


def _encode_categorical_value(base_col: str, value, default_val: float) -> float:
    """Encode a raw categorical value using the stored LabelEncoder."""
    try:
        encoder = _load_label_encoder(base_col)
        encoded_value = encoder.transform([str(value)])[0]
        return float(encoded_value)
    except FileNotFoundError:
        return default_val
    except Exception:
        return default_val


def _load_tabular_ranker():
    """Load XGBoost ranker bundle (model + metadata)."""
    global _RANKER_CACHE
    if _RANKER_CACHE is not None:
        return _RANKER_CACHE
    for model_path in _RANKER_MODEL_PATHS:
        if model_path.exists():
            try:
                bundle = joblib.load(model_path)
            except Exception:
                continue
            if isinstance(bundle, dict) and "model" in bundle and "feature_columns" in bundle:
                _RANKER_CACHE = bundle
                return bundle
    raise FileNotFoundError(f"Ranker tabular no encontrado en rutas: {_RANKER_MODEL_PATHS}.")


def _score_candidates_with_ranker(candidate_indices: np.ndarray, bundle: dict):
    """Return ranking scores and order for candidate indices."""
    feature_cols = bundle.get("feature_columns", [])
    if not feature_cols:
        raise ValueError("El ranker no contiene columnas de características válidas.")
    defaults = bundle.get("defaults", {})
    candidate_df = listings.iloc[candidate_indices].copy()
    missing_cols = [col for col in feature_cols if col not in candidate_df.columns]
    if missing_cols:
        raise ValueError(f"Columnas faltantes en candidatos para el ranker: {missing_cols}")
    fill_values = defaults.copy()
    for col in feature_cols:
        if col not in fill_values:
            base_series = listings[col]
            if base_series.isnull().all():
                fill_values[col] = 0.0
            elif np.issubdtype(base_series.dtype, np.number):
                val = base_series.median(skipna=True)
                if np.isnan(val):
                    val = base_series.mean(skipna=True)
                fill_values[col] = float(val if not np.isnan(val) else 0.0)
            else:
                mode_series = base_series.mode(dropna=True)
                fill_values[col] = float(mode_series.iloc[0]) if not mode_series.empty else 0.0
    feature_matrix = candidate_df[feature_cols].fillna(fill_values).to_numpy(dtype=float)
    scores = bundle["model"].predict(feature_matrix)
    order = np.argsort(scores)[::-1]
    return scores, order


def build_user_vector(query_text: str | None = None,
                      query_attributes: dict | None = None,
                      use_scaled_struct: bool = True):
    """
    Construye el vector del usuario combinando:
      - embedding de texto (si query_text dado) usando `model`
      - vector estructural (llenando ausentes con la media de `listings`)
        para columnas estructurales en el mismo orden que se usan al fusionar items.

    Args:
      query_text: texto del usuario (opcional).
      query_attributes: dict de constraints, claves corresponden a columnas
                        del struct (p.ej. 'price', 'neighbourhood_cleansed', 'property_type', ...).
                        Puedes pasar valores sin codificar; si existe un encoder guardado se aplicará automáticamente.
      use_scaled_struct: si True intenta devolver struct escalado con el mismo StandardScaler usado en entrenamiento.
                        También devuelve la versión sin escalar para compatibilidad.

    Retorna:
      dict con keys:
        - text_emb: embedding (1d numpy) del texto (zeros si no hay texto)
        - raw_struct: vector estructural sin escalar (1d numpy)
        - scaled_struct: vector estructural escalado (1d numpy) o None si no se pudo escalar
        - combined_raw: concatenación [text_emb, raw_struct]
        - combined_scaled: concatenación [text_emb, scaled_struct] (o None)
    """

    struct_cols = list(STRUCT_COLUMNS_FOR_MODEL)
    struct_df = listings[struct_cols]

    query_attributes = query_attributes or {}

    text_dim = text_embeddings.shape[1]
    if query_text:
        user_text_emb = model.encode(query_text, convert_to_numpy=True).reshape(-1)
        if user_text_emb.shape[0] != text_dim:
            user_text_emb = np.resize(user_text_emb, text_dim)
    else:
        user_text_emb = np.zeros(text_dim, dtype=np.float32)

    struct_map = {col.lower(): col for col in struct_cols}
    encoded_map = {col.replace('_encoded', '').lower(): col for col in struct_cols if col.endswith('_encoded')}
    normalized_attrs = {}

    for raw_key, raw_val in query_attributes.items():
        key_lower = str(raw_key).lower()
        if key_lower in struct_map:
            normalized_attrs[struct_map[key_lower]] = raw_val
            continue
        if key_lower in encoded_map:
            normalized_attrs[encoded_map[key_lower]] = raw_val
            continue
        if key_lower.endswith('_encoded'):
            base_key = key_lower.replace('_encoded', '')
            if base_key in encoded_map:
                normalized_attrs[encoded_map[base_key]] = raw_val
                continue

    defaults = {col: _default_for_column(struct_df[col]) for col in struct_cols}

    raw_values = []
    for col in struct_cols:
        if col in normalized_attrs:
            val = normalized_attrs[col]
            if col.endswith('_encoded'):
                if isinstance(val, (int, float, np.integer, np.floating)):
                    raw_values.append(float(val))
                else:
                    base_col = col.replace('_encoded', '')
                    encoded_val = _encode_categorical_value(base_col, val, defaults[col])
                    raw_values.append(encoded_val)
            elif col in RAW_ENCODER_COLUMNS:
                encoded_val = _encode_categorical_value(col, val, defaults[col])
                raw_values.append(encoded_val)
            else:
                try:
                    raw_values.append(float(val))
                except (TypeError, ValueError):
                    raw_values.append(defaults[col])
        else:
            raw_values.append(defaults[col])

    raw_struct = np.asarray(raw_values, dtype=float).reshape(-1)

    scaled_struct = None
    if use_scaled_struct:
        scaler_obj = None
        try:
            if 'scaler' in globals() and hasattr(scaler, 'mean_'):
                scaler_obj = scaler
        except NameError:
            scaler_obj = None

        if scaler_obj is None:
            possible_paths = [
                "/Users/gblasd/Documents/Code/SmartBnB/models/structural_scaler.pkl",
                "/opt/airflow/models/structural_scaler.pkl",
                "../models/structural_scaler.pkl",
                "./models/structural_scaler.pkl",
            ]
            for p in possible_paths:
                if os.path.exists(p):
                    try:
                        scaler_obj = joblib.load(p)
                        break
                    except Exception:
                        scaler_obj = None

        if scaler_obj is not None and hasattr(scaler_obj, 'mean_'):
            try:
                scaled_struct = scaler_obj.transform(raw_struct.reshape(1, -1)).reshape(-1)
            except Exception:
                scaled_struct = None

    combined_raw = np.concatenate([user_text_emb.reshape(-1), raw_struct.reshape(-1)])
    combined_scaled = None
    if scaled_struct is not None:
        combined_scaled = np.concatenate([user_text_emb.reshape(-1), scaled_struct.reshape(-1)])

    return {
        "text_emb": user_text_emb,
        "raw_struct": raw_struct,
        "scaled_struct": scaled_struct,
        "combined_raw": combined_raw,
        "combined_scaled": combined_scaled,
    }


def knn_query(user_text: str = "", user_constraints: dict | None = None, top_k: int = 10,
              use_scaled: bool = True, return_distances: bool = False,
              apply_reranker: bool = True, return_ranking_scores: bool = False):
    """Helper to run KNN search using the stored index con re-ranking opcional."""
    user_vectors = build_user_vector(user_text, user_constraints, use_scaled_struct=use_scaled)

    candidate_vectors = []
    if use_scaled and user_vectors.get("combined_scaled") is not None:
        candidate_vectors.append(("combined_scaled", user_vectors["combined_scaled"]))
    if user_vectors.get("combined_raw") is not None:
        candidate_vectors.append(("combined_raw", user_vectors["combined_raw"]))
    candidate_vectors.append(("text_emb", user_vectors["text_emb"]))  # fallback a solo texto

    knn_paths = [
        "/Users/gblasd/Documents/Code/SmartBnB/models/knn_index.pkl",
        "/opt/airflow/models/knn_index.pkl",
        "../models/knn_index.pkl",
        "./models/knn_index.pkl",
        "/Users/gblasd/Documents/Code/SmartBnB/models/knn_text_embeddings.pkl",  # fallback legacy
    ]

    knn_model = None
    for path in knn_paths:
        if os.path.exists(path):
            try:
                knn_model = joblib.load(path)
                break
            except Exception:
                continue

    if knn_model is None:
        raise FileNotFoundError("No se encontró un índice KNN en las rutas configuradas.")

    expected_dim = getattr(knn_model, "n_features_in_", None)
    usable_vectors = []
    mismatched = []
    for label, vec in candidate_vectors:
        if vec is None:
            continue
        if expected_dim is None or vec.shape[0] == expected_dim:
            usable_vectors.append((label, vec))
        else:
            mismatched.append((label, vec.shape[0]))

    if not usable_vectors:
        dim_msg = f" para dimensión esperada {expected_dim}" if expected_dim is not None else ""
        mismatch_msg = ", ".join([f"{label}: {dim}" for label, dim in mismatched])
        raise ValueError(f"No hay vectores compatibles{dim_msg}. Dimensiones calculadas: {mismatch_msg}.")

    distances = indices = None
    last_error: Exception | None = None
    for label, vec in usable_vectors:
        try:
            distances, indices = knn_model.kneighbors(vec.reshape(1, -1), n_neighbors=top_k, return_distance=True)
            break
        except Exception as exc:
            last_error = exc
            continue

    if indices is None:
        raise ValueError("No fue posible ejecutar knn.kneighbors con los vectores disponibles.") from last_error

    flat_indices = indices[0]
    flat_distances = distances[0] if distances is not None else None
    rerank_scores = None

    if apply_reranker and flat_indices.size > 1:
        try:
            ranker_bundle = _load_tabular_ranker()
            scores, order = _score_candidates_with_ranker(flat_indices, ranker_bundle)
            flat_indices = flat_indices[order]
            if flat_distances is not None:
                flat_distances = flat_distances[order]
            rerank_scores = scores[order]
        except FileNotFoundError:
            print("Ranker tabular no disponible; se conserva el orden del KNN.")
        except Exception as exc:
            print(f"Error al aplicar el ranker tabular: {exc}")

    outputs = [flat_indices]
    if return_distances:
        outputs.append(flat_distances)
    if return_ranking_scores:
        outputs.append(rerank_scores)

    if len(outputs) == 1:
        return outputs[0]
    return tuple(outputs)

In [895]:
# Example usage
sample_query_text = "roof garden cerca del estadio GNP"

indices, scores = knn_query(
    user_text=sample_query_text,
    top_k=20,
    return_ranking_scores=True
)

print("Indices de 20 listings similares (re-ranked):", indices)
print("Scores del ranker tabular:", scores)
print("\nDetalles de los listings ordenados:")
ranked_df = get_listing_by_id(indices)[['id', 'name', 'price', 'description', 'listing_url', 'availability_365']].copy()
ranked_df['rank_score'] = scores
ranked_df

Indices de 20 listings similares (re-ranked): [22094 22084 17201 15258 20384 18600 21492 21489 20015 17431 17459 20652
 19353 20926 20708 17203 21355 18755 19020 20225]
Scores del ranker tabular: [ 6.058384    6.052669    4.348832    0.84878504  0.27925405 -1.6047454
 -4.603419   -4.721714   -4.885199   -5.2080317  -5.584553   -6.7630258
 -7.0225725  -7.3448143  -7.375461   -7.540846   -7.544434   -7.820684
 -7.821686   -7.9604034 ]

Detalles de los listings ordenados:


,id,name,price,description,listing_url,availability_365,rank_score
0,1054369113595006205,Suites amuebladas en San Ángel,470.00,"Lofts apartments, fully furnished, with a priv...",https://www.airbnb.com/rooms/1054369113595006205,173,6.06
1,1162406160618948063,New apartment at Roma Norte CDMX,640.00,Enjoy a stylish experience at this centrally-l...,https://www.airbnb.com/rooms/1162406160618948063,78,6.05
2,1162444052615556841,Departamento Histórico,630.00,None,https://www.airbnb.com/rooms/1162444052615556841,269,4.35
3,1179081249100598607,Hermoso loft con vistas y estacionamiento privado,738.00,Enjoy the simplicity of this quiet accommodati...,https://www.airbnb.com/rooms/1179081249100598607,99,0.85
4,1182688507659529712,Cozy loft Ajusco,510.00,Apartment in Ajusco with a city view. Forget y...,https://www.airbnb.com/rooms/1182688507659529712,252,0.28
5,1243944578710631739,cuarto privado con baño propio y entrada aparte,450.00,Enjoy the simplicity of this quiet and central...,https://www.airbnb.com/rooms/1243944578710631739,88,-1.60
6,1253270143661223519,Hola bonita habitación,675.00,It is close to the forum sun the GNP stadium w...,https://www.airbnb.com/rooms/1253270143661223519,359,-4.60
7,1265443294394297457,Habitación súper céntrica y comunicada,268.00,Super central private room connected to public...,https://www.airbnb.com/rooms/1265443294394297457,142,-4.72
8,1280707869724982834,Habitación céntrica sencilla,252.00,"A simple private room, super central and conne...",https://www.airbnb.com/rooms/1280707869724982834,144,-4.89
9,1318184586582469163,Cama “Rufino Tamayo”,218.00,Enjoy the simplicity of this space in a centra...,https://www.airbnb.com/rooms/1318184586582469163,185,-5.21


In [897]:
idxs, dists, scores = knn_query(
    user_text="Departamentos petfriendly con estacionamiento que este cerca del areopuerto",
    user_constraints={"price": 40000, "room_type": "Entire home/apt"},
    top_k=20,
    return_distances=True,
    return_ranking_scores=True
)

print("Indices re-rankeados:", idxs)
print("Distancias KNN:", dists)
print("Scores del ranker:", scores)
print("\nDetalles de los listings:")
ranked_df = get_listing_by_id(list(idxs))[['id', 'name', 'price', 'description', 'listing_url', 'beds', 'room_type']].copy()
ranked_df['rank_score'] = scores
ranked_df

Indices re-rankeados: [12627 13806 19006 16131 20748 13991 15707 16265 16266 16259 16260 16261
 16249 20342 17508 17476 17505 17514 17549 21672]
Distancias KNN: [0.53121626 0.53134643 0.5312956  0.53123534 0.5313527  0.53134152
 0.53136368 0.5312142  0.5312141  0.5312141  0.53121417 0.53121448
 0.53121398 0.53136095 0.53121424 0.53121418 0.53121467 0.53121446
 0.5312145  0.53121404]
Scores del ranker: [ 3.573818  -5.3695545 -7.7011404 -7.820029  -8.320841  -8.323855
 -8.325446  -8.539071  -8.539071  -8.547664  -8.547664  -8.617113
 -8.617113  -8.712777  -8.874738  -8.874738  -8.874738  -8.888989
 -8.90942   -8.929253 ]

Detalles de los listings:


,id,name,price,description,listing_url,beds,room_type,rank_score
0,892676742760732948,¡Recámara acogedora y muy cómoda!,900000.00,"Comfortable, spacious rental bedroom, inside t...",https://www.airbnb.com/rooms/892676742760732948,1,Castle,3.57
1,963204479375809359,3 Thoughtfully curated Villas / Heart of Condesa,55206.00,Welcome to our fabulous fully equipped houses...,https://www.airbnb.com/rooms/963204479375809359,22,Campsite,-5.37
2,973888328897033864,3 Thoughtfully curated Villas / Heart of Condesa,55206.00,Welcome to our fabulous fully equipped houses...,https://www.airbnb.com/rooms/973888328897033864,22,Campsite,-7.70
3,1083472398314386316,3 Thoughtfully curated Villas,51000.00,Welcome to our 3 fabulous fully equipped hou...,https://www.airbnb.com/rooms/1083472398314386316,25,Campsite,-7.82
4,1106549995939886571,Gorgeous Suite alongside Masaryk,99000.00,This place has a strategic location - it will ...,https://www.airbnb.com/rooms/1106549995939886571,1,Campsite,-8.32
5,1118120677165122021,"Pets Welcome, Spa - Near El Pendulo Bookstore!",759762.00,The hotel is situated in a lively metropolis i...,https://www.airbnb.com/rooms/1118120677165122021,1,Casa particular,-8.32
6,1118172352853118628,Surreal Design Escape in 3 Units at La Condesa,759762.00,The hotel is situated in a lively metropolis i...,https://www.airbnb.com/rooms/1118172352853118628,3,Casa particular,-8.33
7,1118173216727497991,3 Std King Rooms w/ Spa - Steps from Parque Mé...,759762.00,The hotel is situated in a lively metropolis i...,https://www.airbnb.com/rooms/1118173216727497991,3,Casa particular,-8.54
8,1118175277022551558,Dive into Casa Lamm Culture w/ Shuttle & Night...,759762.00,The hotel is situated in a lively metropolis i...,https://www.airbnb.com/rooms/1118175277022551558,1,Casa particular,-8.54
9,1118176220387018952,Mexico City Luxury Oasis with Pool | 2 Units,759762.00,The hotel is situated in a lively metropolis i...,https://www.airbnb.com/rooms/1118176220387018952,2,Casa particular,-8.55


In [898]:
similar_listing_indices = knn_query(
    user_text="roof cerca del estadio GNP", 
    user_constraints={
        "price": 1500, 
        'availability_365':20
        }, 
    top_k=20)

print("Indices of 20 similar listings:", similar_listing_indices)
print("\nDetails of similar listings:")
get_listing_by_id(similar_listing_indices)[['id', 'name', 'price', 'description', 'listing_url', 'beds', 'availability_365']]

Indices of 20 similar listings: [4339 3450 4786 1069 1666 7181 7141 5759 4378 6831 7008 7193 4956 7094
 7725 1693 5739 3110 5700 7158]

Details of similar listings:


,id,name,price,description,listing_url,beds,availability_365
0,16136153,Linares,267.00,It is a well-lit apartment with a balcony faci...,https://www.airbnb.com/rooms/16136153,1,0
1,19909385,"Amplia Habitación con baño, centro Coyoacan y ...",171.00,"Comfortable room with bathroom, fully equipped...",https://www.airbnb.com/rooms/19909385,1,0
2,20016824,"Private Room In Family Apartment, Mexico City.",272.00,"Private guest house, very quiet area, Wifi, ca...",https://www.airbnb.com/rooms/20016824,1,0
3,29834040,Departamento Eunonia,270.00,"Excellent room in the Narvarte, super connecte...",https://www.airbnb.com/rooms/29834040,1,0
4,32184659,Habitación privada muy cómoda en zona Sur,288.00,Very comfortable and pleasant private room in ...,https://www.airbnb.com/rooms/32184659,1,0
5,37541607,Habitación grande e iluminada,267.00,Spacious space with guests from all over the w...,https://www.airbnb.com/rooms/37541607,1,0
6,37659118,Pily Room 1 PB,210.00,"Excellent location, close to strategic points ...",https://www.airbnb.com/rooms/37659118,2,0
7,39923707,Pily Room 2,189.00,"Excellent location, close to strategic points ...",https://www.airbnb.com/rooms/39923707,1,0
8,41258700,Cuarto céntrico en la azotea cerca de Coyoacán,190.00,It is a very safe room for 2 people on the roo...,https://www.airbnb.com/rooms/41258700,1,0
9,44596671,Amplio departamento equipado y seguro en Santa Fe,3300.00,"66m2 apartment overlooking the Santa Fe area, ...",https://www.airbnb.com/rooms/44596671,3,249


In [899]:
similar_listing_indices = knn_query(
    user_text="Quiero un listing pet-friendly con terraza y cerca del metro",
    top_k=20)

print("Indices of 20 similar listings:", similar_listing_indices)
print("\nDetails of similar listings:")
get_listing_by_id(similar_listing_indices)[['id', 'name', 'price', 'description', 'listing_url', 'beds', 'availability_365']]

Indices of 20 similar listings: [22182 22084 19421 19790 17958 18257 22615 18600 22619 21779 18026 20015
 16470 16308 19353 18514 19020 20225 15665 15976]

Details of similar listings:


,id,name,price,description,listing_url,beds,availability_365
0,1079323346549332893,Suite independiente en CDMX,593.00,"It has an excellent location, connected to imp...",https://www.airbnb.com/rooms/1079323346549332893,2,329
1,1102269576875607401,Céntrico y cómodo departamento,537.00,"Comfortable, pleasant and central department ...",https://www.airbnb.com/rooms/1102269576875607401,1,142
2,1115339598931051833,Habitación baño exclusivo excelente ubicación ...,376.00,It is located very close to the historic cente...,https://www.airbnb.com/rooms/1115339598931051833,1,41
3,1126300293722383326,Men Only Gay-Friendly Shared Cozy,429.00,"A private room in a shared apartment, located ...",https://www.airbnb.com/rooms/1126300293722383326,1,363
4,1211146356158252869,Cozy Private Room in Prime Location,360.00,This tidy and Private Room it's the perfect ho...,https://www.airbnb.com/rooms/1211146356158252869,1,353
5,1212642540407320197,Habitación grande en hermosa casa Mid Century,528.00,Beautiful private room in a mid-century house ...,https://www.airbnb.com/rooms/1212642540407320197,1,80
6,1221967607946043094,Estudio en Tlatelolco,452.00,"Perfectly located accommodation, with 3 metrob...",https://www.airbnb.com/rooms/1221967607946043094,1,54
7,1237786791338432248,Petit Studio,696.00,"Full accommodation with independent access, on...",https://www.airbnb.com/rooms/1237786791338432248,1,157
8,1243944578710631739,cuarto privado con baño propio y entrada aparte,450.00,Enjoy the simplicity of this quiet and central...,https://www.airbnb.com/rooms/1243944578710631739,1,88
9,1265443294394297457,Habitación súper céntrica y comunicada,268.00,Super central private room connected to public...,https://www.airbnb.com/rooms/1265443294394297457,1,142


In [875]:
get_listing_by_id(similar_listing_indices).columns

Index(['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
       'host_since_delta', 'host_response_rate', 'host_acceptance_rate',
       'host_total_listings_count', 'host_listings_count', 'price',
       'accommodates', 'bathrooms', 'bedrooms', 'beds', 'property_type',
       'amenities_n', 'room_type', 'calculated_host_listings_count',
       'calculated_host_listings_count_entire_homes',
       'calculated_host_listings_count_private_rooms',
       'calculated_host_listings_count_shared_rooms', 'number_of_reviews',
       'reviews_per_month', 'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'number_of_reviews_ltm',
       'number_of_reviews_l30d', 'first_review', 'last_review',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'minimum_nights',
       'maximum_nights', 'minimum_minimum_nights', 'maxim